# Data Retrieval

In [3]:
import requests
from bs4 import BeautifulSoup
from chembl_webresource_client.new_client import new_client
import pandas as pd
import json

/Users/vlad/Documents/University/Master-MIND/DALAS-Project/.venv/lib/python3.11/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


## ChEMBL

Python API client tutorial:
https://hub.2i2c.mybinder.org/user/chembl-chembl_webresource_client-7wkmmjs9/notebooks/demo_wrc.ipynb


In [4]:
molecule = new_client.molecule
drug_indication = new_client.drug_indication
target = new_client.target
activity = new_client.activity

### Get MeSH IDs of Autoimmune diseases

In [5]:
# Exctract all MeSH terms of autoimmune diseases
# Site with MeSH terms with autoimmune diseases
url = "https://www.ncbi.nlm.nih.gov/mesh?Db=mesh&Cmd=DetailsSearch&Term=%22Autoimmune+Diseases%22%5BMeSH+Terms%5D"
response = requests.get(url)
if response.status_code == 200:
    html_content = response.text
    soup = BeautifulSoup(html_content, "html.parser")

soup.find_all("span", string="Autoimmune Diseases")
autoimm_ul = soup.find("span", string="Autoimmune Diseases").find_all_next("ul")
autoimm_ul[8].find_all("a")[0].get("href")

mesh_ids = []

for disease_a in autoimm_ul[8].find_all("a"):
    disease_url = disease_a.get('href')
    response = requests.get(f'https://www.ncbi.nlm.nih.gov{disease_url}')
    if response.status_code == 200:
        html_content = response.text
        disease_soup = BeautifulSoup(html_content, "html.parser")
        mesh_ids.append(disease_soup.find("p", string=lambda text: text and text.startswith("MeSH Unique ID:")).text.split()[-1])

In [6]:
print(f'Autoimmune diseases: {len(mesh_ids)}')

Autoimmune diseases: 50


In [7]:
autoimmune_ind = drug_indication.filter(mesh_id__in=mesh_ids)
indications_df = pd.DataFrame(autoimmune_ind)
chembl_ids = pd.unique(indications_df["molecule_chembl_id"]).tolist()
autoimmune_drugs = molecule.filter(molecule_chembl_id__in = chembl_ids, max_phase__gte = 3)
drugs_df = pd.DataFrame(autoimmune_drugs)

In [8]:
print(f'Drug-disease indications: {len(indications_df)}')
print(f'Drugs used for autoimmune diseses: {len(drugs_df)}')
print(f'Biologics: {drugs_df["biotherapeutic"].notnull().sum()}')
print(f'Small molecules: {drugs_df["biotherapeutic"].isnull().sum()}')

Drug-disease indications: 1256
Drugs used for autoimmune diseses: 515
Biologics: 125
Small molecules: 390


In [9]:
matches_df = pd.merge(indications_df, drugs_df, on="molecule_chembl_id")

In [10]:
drugs_df

,atc_classifications,availability_type,biotherapeutic,black_box_warning,chemical_probe,chirality,cross_references,dosed_ingredient,first_approval,first_in_class,...,prodrug,structure_type,therapeutic_flag,topical,usan_stem,usan_stem_definition,usan_substem,usan_year,veterinary,withdrawn_flag
0,"[C01EB03, M01AB01, M01AB51, M02AA23, S01BC01]",1.0,None,1,0,2,"[{'xref_id': 'indomethacin', 'xref_name': 'ind...",True,1965.0,0,...,0,MOL,True,True,None,None,None,1963.0,0,False
1,"[J01MA02, S01AE03, S02AA15, S03AA07]",1.0,None,1,0,2,"[{'xref_id': 'ciprofloxacin', 'xref_name': 'ci...",True,1987.0,0,...,0,MOL,True,True,-oxacin,antibacterials (quinolone derivatives),-oxacin,1987.0,0,False
2,[N05BA01],1.0,None,1,0,2,"[{'xref_id': 'diazepam', 'xref_name': 'diazepa...",True,1963.0,0,...,0,MOL,True,True,-azepam,antianxiety agents (diazepam type),-azepam,1963.0,0,False
3,"[L01EG04, L04AH01, S01XA23]",1.0,None,1,1,1,"[{'xref_id': 'sirolimus', 'xref_name': 'siroli...",True,1999.0,0,...,0,MOL,True,True,-imus,immunosuppressives,-imus,1993.0,0,False
4,[],1.0,None,1,1,1,[],False,1994.0,0,...,0,MOL,True,True,-imus,immunosuppressives,-imus,1992.0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
510,[],1.0,None,1,0,1,"[{'xref_id': 'human/EPAR/zilbrysq', 'xref_name...",True,2023.0,0,...,0,NONE,True,False,-coplan,complement ligand inhibitors,-coplan,NaN,0,False
511,[],1.0,None,1,0,1,"[{'xref_id': 'upadacitinib%20hemihydrate', 'xr...",True,2019.0,0,...,0,MOL,True,False,-tinib,tyrosine kinase inhibitors: tyrosine kinase in...,-tinib (-citinib),NaN,0,False
512,[L01XX27],1.0,None,1,0,2,"[{'xref_id': 'arsenic%20trioxide', 'xref_name'...",True,2000.0,0,...,0,MOL,True,False,None,None,None,2001.0,0,False
513,[],1.0,None,1,0,2,[],True,2023.0,0,...,0,MOL,True,False,-trombopag,thrombopoetin agonists,-trombopag,NaN,0,False


In [11]:
indications_df

,drugind_id,efo_id,efo_term,indication_refs,max_phase_for_ind,mesh_heading,mesh_id,molecule_chembl_id,parent_molecule_chembl_id
0,22607,EFO:0000685,rheumatoid arthritis,"[{'ref_id': 'NCT00048568,NCT00048581,NCT000489...",4.0,"Arthritis, Rheumatoid",D001172,CHEMBL1201823,CHEMBL1201823
1,22633,EFO:0000685,rheumatoid arthritis,"[{'ref_id': 'NCT00049751,NCT00195650,NCT001956...",4.0,"Arthritis, Rheumatoid",D001172,CHEMBL1201580,CHEMBL1201580
2,22744,EFO:0004991,Myasthenia gravis,"[{'ref_id': 'NCT01727193', 'ref_type': 'Clinic...",3.0,Myasthenia Gravis,D009157,CHEMBL1542,CHEMBL1542
3,22837,EFO:0002609,juvenile idiopathic arthritis,"[{'ref_id': 'NCT00426218', 'ref_type': 'Clinic...",4.0,"Arthritis, Juvenile",D001171,CHEMBL1201834,CHEMBL1201834
4,22868,EFO:0002609,juvenile idiopathic arthritis,"[{'ref_id': 'NCT00652925,NCT00807846', 'ref_ty...",4.0,"Arthritis, Juvenile",D001171,CHEMBL118,CHEMBL118
...,...,...,...,...,...,...,...,...,...
1251,156753,MONDO:0007915,systemic lupus erythematosus,"[{'ref_id': 'ZOLACABTAGENE AUTOLEUCEL', 'ref_t...",1.0,"Lupus Erythematosus, Systemic",D008180,CHEMBL6068475,CHEMBL6068475
1252,156910,MONDO:0007915,systemic lupus erythematosus,"[{'ref_id': 'NCT01405196', 'ref_type': 'Clinic...",2.0,"Lupus Erythematosus, Systemic",D008180,CHEMBL6068160,CHEMBL6068160
1253,157057,EFO:1001466,Graves ophthalmopathy,"[{'ref_id': 'NCT02393183', 'ref_type': 'Clinic...",2.0,Graves Ophthalmopathy,D049970,CHEMBL6068503,CHEMBL6068503
1254,157176,MONDO:0005147,type 1 diabetes mellitus,"[{'ref_id': 'NCT03272269', 'ref_type': 'Clinic...",1.0,"Diabetes Mellitus, Type 1",D003922,CHEMBL6068583,CHEMBL6068583


### Targets and Mechanisms

TODO: get lists of molecular targets / mechanisms for each drugs

It will be useful for graph models

In [12]:
target = new_client.target
mechanism = new_client.mechanism

In [13]:
drug_mech = mechanism.filter(molecule_chembl_id__in = chembl_ids[2])
print(f'{len(drug_mech)=}')
print(drug_mech[0])
print(drug_mech[1])

len(drug_mech)=2
{'action_type': 'INHIBITOR', 'binding_site_comment': None, 'direct_interaction': 1, 'disease_efficacy': 1, 'max_phase': 4, 'mec_id': 1744, 'mechanism_comment': 'Prodrug enzyme: hypoxanthine-guanine phosphoribosyl transferase (HGPRT; EC 2.4.2.8)', 'mechanism_of_action': 'Amidophosphoribosyltransferase inhibitor', 'mechanism_refs': [{'ref_id': '18506437', 'ref_type': 'PubMed', 'ref_url': 'http://europepmc.org/abstract/MED/18506437'}], 'molecular_mechanism': 1, 'molecule_chembl_id': 'CHEMBL1542', 'parent_molecule_chembl_id': 'CHEMBL1542', 'record_id': 1343926, 'selectivity_comment': None, 'site_id': None, 'target_chembl_id': 'CHEMBL2362992', 'variant_sequence': None}
{'action_type': 'INHIBITOR', 'binding_site_comment': None, 'direct_interaction': 1, 'disease_efficacy': 1, 'max_phase': 4, 'mec_id': 1874, 'mechanism_comment': 'Prodrug enzyme: hypoxanthine-guanine phosphoribosyl transferase (HGPRT; EC 2.4.2.8)', 'mechanism_of_action': 'DNA inhibitor', 'mechanism_refs': [{'re

In [14]:
drug_targets = target.filter(molecule_chembl_id__in = chembl_ids[0])
print(f'{len(drug_targets)=}')
drug_targets[0]

len(drug_targets)=17803


{'cross_references': [],
 'organism': 'Homo sapiens',
 'pref_name': 'Maltase-glucoamylase',
 'species_group_flag': False,
 'target_chembl_id': 'CHEMBL2074',
 'target_components': [{'accession': 'O43451',
   'component_description': 'Maltase-glucoamylase',
   'component_id': 434,
   'component_type': 'PROTEIN',
   'relationship': 'SINGLE PROTEIN',
   'target_component_synonyms': [{'component_synonym': '3.2.1.20',
     'syn_type': 'EC_NUMBER'},
    {'component_synonym': 'Alpha-1,4-glucosidase', 'syn_type': 'UNIPROT'},
    {'component_synonym': 'Maltase-glucoamylase', 'syn_type': 'UNIPROT'},
    {'component_synonym': 'MGA', 'syn_type': 'GENE_SYMBOL_OTHER'},
    {'component_synonym': 'MGAM', 'syn_type': 'GENE_SYMBOL'},
    {'component_synonym': 'MGAML', 'syn_type': 'GENE_SYMBOL_OTHER'},
    {'component_synonym': 'Synonyms=MGA', 'syn_type': 'GENE_SYMBOL_OTHER'}],
   'target_component_xrefs': [{'xref_id': 'O43451',
     'xref_name': None,
     'xref_src_db': 'AlphaFoldDB'},
    {'xref_id': '

## Disease Involved Molecules and Mechanisms

TODO: find a nice database and match target / mechnisms IDs with drugs

## Clinical Trials Metadata

TODO: explore further the variables proposed 

Documentation: https://clinicaltrials.gov/data-api/api

API Server: https://clinicaltrials.gov/api/v2

In [ ]:
# TODO: get NCT IDs for all indications and get their timestamps, 
# for indictions from ChEMBL, I guess that we need only timestamps 
# and maybe some other metadata, but we consider them sucessfull by default 
# (otherwise they will not appear in indications)

# TODO: so we need to do the same thing but for all pairs drug - disease 
# using queries and filters of ClinicalTrials API and access the global outcome
# - success, failure, unknown

test_row = 0
ind_refs = indications_df["indication_refs"][test_row]
for ref in ind_refs:
    if ref["ref_type"] == "ClinicalTrials":
        ref_ids = ref["ref_id"].split(",")
ref_ids[:5]

['NCT00048568', 'NCT00048581', 'NCT00048932', 'NCT00095147', 'NCT00122382']

In [ ]:
test_ref_id = ref_ids[0]
# TODO: maybe get all trials at ones with a single query 
# searching in list of all ids with `query.id` but will need to handle pagination

# TODO: request only needed fields
# e.g. https://clinicaltrials.gov/api/v2/studies?query=trastuzumab&fields=protocol_section.status_module.overall_status,results_section.outcome_measure_list.outcome_measure.analysis_list.analysis.p_value

nct_url = f"https://clinicaltrials.gov/api/v2/studies/{test_ref_id}?format=json"
nct_response = requests.get(nct_url)
nct_info = nct_response.json()
print(json.dumps(nct_info, indent=2))

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT00048568",
      "orgStudyIdInfo": {
        "id": "IM101-102"
      },
      "organization": {
        "fullName": "Bristol-Myers Squibb",
        "class": "INDUSTRY"
      },
      "briefTitle": "A Phase III Study of Abatacept (BMS-188667) in Patients With Active Rheumatoid Arthritis and Inadequate Response to Methotrexate",
      "officialTitle": "A Phase 3, Multi-Center, Randomized, Double-Blind, Placebo-Controlled Study and Open-Label Study to Evaluate the Efficacy and Safety of Abatacept in Combination Therapy With Methotrexate Versus Methotrexate Alone in Subjects With Active Rheumatoid Arthritis and Inadequate Response to Methotrexate"
    },
    "statusModule": {
      "statusVerifiedDate": "2011-10",
      "overallStatus": "COMPLETED",
      "expandedAccessInfo": {
        "hasExpandedAccess": false
      },
      "startDateStruct": {
        "date": "2002-12"
      },
      "primaryCompletionDateStruct

In [51]:
# contains phase
print(json.dumps(nct_info["protocolSection"]["designModule"], indent=2)) 

{
  "studyType": "INTERVENTIONAL",
  "phases": [
    "PHASE3"
  ],
  "designInfo": {
    "allocation": "RANDOMIZED",
    "interventionModel": "PARALLEL",
    "primaryPurpose": "TREATMENT",
    "maskingInfo": {
      "masking": "DOUBLE",
      "whoMasked": [
        "PARTICIPANT",
        "INVESTIGATOR"
      ]
    }
  },
  "enrollmentInfo": {
    "count": 1250,
    "type": "ACTUAL"
  }
}


In [52]:
# contains timestamp and status
# interested by COMPLETED status, handle TERMINATED cases - ignore or check reasons why
print(json.dumps(nct_info["protocolSection"]["statusModule"] , indent=2)) 

{
  "statusVerifiedDate": "2011-10",
  "overallStatus": "COMPLETED",
  "expandedAccessInfo": {
    "hasExpandedAccess": false
  },
  "startDateStruct": {
    "date": "2002-12"
  },
  "primaryCompletionDateStruct": {
    "date": "2009-10",
    "type": "ACTUAL"
  },
  "completionDateStruct": {
    "date": "2009-10",
    "type": "ACTUAL"
  },
  "studyFirstSubmitDate": "2002-11-02",
  "studyFirstSubmitQcDate": "2002-11-12",
  "studyFirstPostDateStruct": {
    "date": "2002-11-13",
    "type": "ESTIMATED"
  },
  "resultsFirstSubmitDate": "2011-03-28",
  "resultsFirstSubmitQcDate": "2011-10-26",
  "resultsFirstPostDateStruct": {
    "date": "2011-12-05",
    "type": "ESTIMATED"
  },
  "lastUpdateSubmitDate": "2011-10-26",
  "lastUpdatePostDateStruct": {
    "date": "2011-12-05",
    "type": "ESTIMATED"
  }
}


In [ ]:
# can use a hearistic on the success of the trial - e.g. median p.value :)

# maybe this strategy:
#   success: status==COMPLETED and pvalues ok 
#   failure: status == COMPLETED and pvalues not ok
#   unknown: status != COMPLETED or no results (but maybe need to find out why not completed)

if nct_info["hasResults"]:
    measures = nct_info["resultsSection"]["outcomeMeasuresModule"]["outcomeMeasures"]
    for measure in measures:
        if "analyses" in measure:
            print(measure["analyses"][0]["pValue"])
else:
    # TODO: why sometimes no results
    print("No results")

<0.001
<0.001
0.029
<0.001
<0.001
<0.001
<0.001
<0.001
<0.001
<0.001
<0.001
0.002


## Embeddings Extraction

To create a score o semantic proximity between drugs and diseases.

Can compare only embeddings of names of drugs and diseases or full textual descriptions (e.g. DrugBank or CheMBL description for drugs and  MeSH or PubMed summary for diseases) using cosine similrity.

SapBERT (for concept-level similarity) or PubMedBERT (for descriptive embeddings)

maybe Bio-Bert?

all from `transformers` module

Can do both word and description embeddings (but compare only those of the same kind)

